In [3]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path

# ==========================
# FOLDER
# ==========================

folder="."

files=sorted(
    Path(folder)
    .glob("cleaned_*.xlsx")
)

print("Files Found:",len(files))

# ==========================
# DATE FORMAT DETECTOR
# ==========================

def detect_date_format(series):

    sample=(
        series
        .dropna()
        .astype(str)
        .str.strip()
    )

    sample=sample[
        sample!=""
    ]

    sample=sample.head(50)

    if len(sample)==0:
        return ""

    patterns=[]

    for v in sample:

        if re.match(
            r"^\d{2}-[A-Za-z]{3}-\d{2}$",
            v
        ):
            patterns.append(
                "dd-Mon-yy"
            )

        elif re.match(
            r"^\d{4}-\d{2}-\d{2}",
            v
        ):
            patterns.append(
                "yyyy-mm-dd"
            )

        elif re.match(
            r"^\d{2}/\d{2}/\d{4}$",
            v
        ):
            patterns.append(
                "dd/mm/yyyy"
            )

        elif re.match(
            r"^\d{2}-\d{2}-\d{4}$",
            v
        ):
            patterns.append(
                "dd-mm-yyyy"
            )

        elif re.match(
            r"^[A-Za-z]{3}$",
            v
        ):
            patterns.append(
                "month_only"
            )

    if patterns:
        return pd.Series(
            patterns
        ).mode()[0]

    return ""

# ==========================
# ANALYZE FILES
# ==========================

report=[]

for file in files:

    print("\nChecking:",file.name)

    df=pd.read_excel(
        file,
        dtype=str
    )

    for col in df.columns:

        dtype=str(
            df[col].dtype
        )

        missing=round(
            (
                df[col]
                .isna()
                .mean()
            )*100,
            2
        )

        sample=(
            df[col]
            .dropna()
            .astype(str)
            .head(1)
        )

        sample=(
            sample.iloc[0]
            if len(sample)
            else ""
        )

        date_format=""

        if "date" in col.lower():

            date_format=(
                detect_date_format(
                    df[col]
                )
            )

        report.append([

            file.name,
            col,
            dtype,
            missing,
            sample,
            date_format

        ])

# ==========================
# EXPORT REPORT
# ==========================

report_df=pd.DataFrame(
    report,
    columns=[

        "File",
        "Column",
        "Datatype",
        "Missing_%",
        "Sample_Value",
        "Date_Format"

    ]
)

output="near_miss_schema_report.xlsx"

report_df.to_excel(
    output,
    index=False
)

print("\nSaved:",output)

print("\nUnique Datatypes:")
print(
    report_df["Datatype"]
    .value_counts()
)

Files Found: 12

Checking: cleaned_10_Near_Miss.xlsx

Checking: cleaned_11_Near_Miss.xlsx

Checking: cleaned_12_Near_Miss.xlsx

Checking: cleaned_1_Near_Miss.xlsx

Checking: cleaned_2_Near_Miss.xlsx

Checking: cleaned_3_Near_Miss.xlsx

Checking: cleaned_4_Near_Miss.xlsx

Checking: cleaned_5_Near_Miss.xlsx

Checking: cleaned_6_Near_Miss.xlsx

Checking: cleaned_7_Near_Miss.xlsx

Checking: cleaned_8_Near_Miss.xlsx

Checking: cleaned_9_Near_Miss.xlsx

Saved: near_miss_schema_report.xlsx

Unique Datatypes:
Datatype
object    302
Name: count, dtype: int64
